In [ ]:

from __future__ import annotations

import os
import re
from itertools import combinations
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Keep vector text editable in SVG/PDF outputs.
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["xtick.major.width"] = 0.8
plt.rcParams["ytick.major.width"] = 0.8


# ============================================================
# User configuration
# ============================================================

# Edit this if you are not running the notebook from the evaluation root.
# The root should contain folders like:
#   Replogle_K562_essential_pseudo_pairing_evaluation/single/
root_dir = Path(os.getcwd())

# Datasets and perturbation groups to scan.
dataset_names = [
    "Replogle_K562_essential",
    "Replogle_RPE",
    "NormanWeissman2019",
    "ChangYe",
    "ZhaoSims2021",
]

groups = ["single", "dual", "multi"]

# If True, process every existing dataset/group under root_dir.
# If False, only process SELECTED_SUB_PATH.
RUN_ALL_VALID_PATHS = True
SELECTED_SUB_PATH: Path | None = None

# Whether to keep S0 if it is selected in selected_variants_TEMPLATE_EDIT_ME.csv.
# Usually S0 is a reference rather than a selected strategy bar.
INCLUDE_S0_AS_BAR = False

# Output folder under each result_analysis folder.
OUTPUT_FOLDER_NAME = "selected_inverse_mlp_barplots"

# Save / display controls.
SAVE_PNG = True
SAVE_SVG = True
SHOW_FIGURES = True
DPI = 300


# Replicate-level plotting controls.
# Five dots can only be drawn from a run/seed-level table. A seed-averaged table
# contains one mean row per variant and therefore cannot recover the five values.
EXPECTED_REPLICATIONS_PER_VARIANT = 5
REQUIRE_REPLICATE_LEVEL_TABLE = True



# ============================================================
# Metric configuration
# ============================================================

INVERSE_MLP_METRICS = {
    "mlp_inverse__test_accuracy_mean": {
        "label": "Inverse MLP test accuracy",
        "ylabel": "Test accuracy",
        "direction": "higher",
        "save_name": "inverse_mlp_test_accuracy_barplot",
        "value_format": ".4f",
    },
    "mlp_inverse__macro_auc_mean": {
        "label": "Inverse MLP macro AUC",
        "ylabel": "Macro AUC",
        "direction": "higher",
        "save_name": "inverse_mlp_macro_auc_barplot",
        "value_format": ".4f",
    },
    "mlp_inverse__macro_f1_mean": {
        "label": "Inverse MLP macro F1",
        "ylabel": "Macro F1",
        "direction": "higher",
        "save_name": "inverse_mlp_macro_f1_barplot",
        "value_format": ".4f",
    }, 
    "mlp_inverse__recall_mean": {
        "label": "Inverse MLP recall",
        "ylabel": "Recall",
        "direction": "higher",
        "save_name": "inverse_mlp_recall_barplot",
        "value_format": ".4f",
    }, 
    "mlp_inverse__precision_mean": {
        "label": "Inverse MLP precision",
        "ylabel": "Precision",
        "direction": "higher",
        "save_name": "inverse_mlp_precision_barplot",
        "value_format": ".4f",
    }, 
}

# Source-column fallback if the canonical metric columns are absent.
# The notebook fills the canonical columns above from any of these alternatives.
METRIC_SOURCE_FALLBACKS = {
    "mlp_inverse__test_accuracy_mean": [
        "mlp_inverse__test_accuracy_mean",
        "mlp_inverse__test_accuracy",
        "inverse_mlp__test_accuracy_mean",
        "inverse_mlp__test_accuracy",
        "test_accuracy_mean",
        "test_accuracy",
        "accuracy_mean",
        "accuracy",
    ],
    "mlp_inverse__macro_auc_mean": [
        "mlp_inverse__macro_auc_mean",
        "mlp_inverse__macro_auc",
        "inverse_mlp__macro_auc_mean",
        "inverse_mlp__macro_auc",
        "macro_auc_mean",
        "macro_auc",
        "test_macro_auc_mean",
        "test_macro_auc",
        "test_macro_auc_ovr",
        "test_macro_auc_ovr_mean",
    ],
    "mlp_inverse__macro_f1_mean": [
        "mlp_inverse__macro_f1_mean",
        "mlp_inverse__macro_f1",
        "inverse_mlp__macro_f1_mean",
        "inverse_mlp__macro_f1",
        "macro_f1_mean",
        "macro_f1",
        "test_macro_f1_mean",
        "test_macro_f1",
    ],
    "mlp_inverse__recall_mean": [
        "mlp_inverse__recall_mean",
        "mlp_inverse__recall",
        "inverse_mlp__recall_mean",
        "inverse_mlp__recall",
        "recall_mean",
        "recall",
        "test_recall_mean",
        "test_recall",
        "test_macro_recall",
        "macro_recall",
    ],
    "mlp_inverse__precision_mean": [
        "mlp_inverse__precision_mean",
        "mlp_inverse__precision",
        "inverse_mlp__precision_mean",
        "inverse_mlp__precision",
        "precision_mean",
        "precision",
        "test_precision_mean",
        "test_precision",
        "test_macro_precision",
        "macro_precision",
    ],
}

# Fallback std columns for summary-only tables.
METRIC_STD_FALLBACKS = {
    "mlp_inverse__test_accuracy_mean": [
        "mlp_inverse__test_accuracy_std",
        "inverse_mlp__test_accuracy_std",
        "test_accuracy_std",
        "accuracy_std",
        "mlp_inverse__test_accuracy_sd",
        "test_accuracy_sd",
    ],
    "mlp_inverse__macro_auc_mean": [
        "mlp_inverse__macro_auc_std",
        "inverse_mlp__macro_auc_std",
        "macro_auc_std",
        "test_macro_auc_std",
        "mlp_inverse__macro_auc_sd",
        "macro_auc_sd",
    ],
    "mlp_inverse__macro_f1_mean": [
        "mlp_inverse__macro_f1_std",
        "inverse_mlp__macro_f1_std",
        "macro_f1_std",
        "test_macro_f1_std",
        "mlp_inverse__macro_f1_sd",
        "macro_f1_sd",
    ],
    "mlp_inverse__recall_mean": [
        "mlp_inverse__recall_std",
        "inverse_mlp__recall_std",
        "recall_std",
        "test_recall_std",
        "mlp_inverse__recall_sd",
        "recall_sd",
    ],
    "mlp_inverse__precision_mean": [
        "mlp_inverse__precision_std",
        "inverse_mlp__precision_std",
        "precision_std",
        "test_precision_std",
        "mlp_inverse__precision_sd",
        "precision_sd",
    ],
}

# Clear y-limit controls.
# Default: fix the lower bound at 0 and auto-compute the upper bound so error bars, dots, and annotations are not clipped.
# To force bounded score axes, set the top to 1.0 or 1.05 manually.
Y_LIM_BY_METRIC = {
    "mlp_inverse__test_accuracy_mean": (0.0, None),
    "mlp_inverse__macro_auc_mean": (0.70, None),
    "mlp_inverse__macro_f1_mean": (0.0, None),
    "mlp_inverse__recall_mean": (0.0, None),
    "mlp_inverse__precision_mean": (0.0, None),
}

# Optional dataset/group/metric-specific override.
# Example:
# Y_LIM_BY_DATASET_GROUP_METRIC = {
#     ("Replogle_K562_essential", "single", "mlp_inverse__test_accuracy_mean"): (0.0, 1.0),
# }
Y_LIM_BY_DATASET_GROUP_METRIC = {}

# Margin used only when one side of the y-limit is None.
Y_MARGIN_FRACTION = 0.08



# ============================================================
# Plot style controls copied/adapted from the attached bar-plot notebook
# ============================================================

# S0 is shown as a dashed horizontal reference line, not as a bar.
NAIVE_BASELINE_ID = "S0_naive_mean_control_reference"
SHOW_NAIVE_BASELINE_LINE = True
NAIVE_BASELINE_LINE_COLOR = "#EE5862"
NAIVE_BASELINE_LINESTYLE = "--"
NAIVE_BASELINE_LINEWIDTH = 1.15
NAIVE_BASELINE_LINE_ALPHA = 0.85
NAIVE_BASELINE_TEXT = "Naive average control"
NAIVE_BASELINE_TEXT_SIZE = 10
NAIVE_BASELINE_TEXT_COLOR = "#555555"
NAIVE_BASELINE_TEXT_X_FRACTION = 0.985
NAIVE_BASELINE_TEXT_Y_OFFSET_FRACTION = 0.012

# Significance is tested against random single control only, excluding S0.
REFERENCE_STRATEGY_FOR_TEST = "S1_random_single_control"
REFERENCE_STRATEGY_FOR_TEST_LABEL = "Random single control"
REFERENCE_BAR_TEXT = "ref"
SHOW_REFERENCE_BAR_TEXT = False

# Use strategy labels for bar x-axis labels.
USE_STRATEGY_PLOT_LABELS_FOR_XTICKS = True
ADD_VARIANT_SUFFIX_TO_XTICKS = True

# Statistics: "auto", "paired", or "unpaired".
# auto uses paired tests when every selected variant has the same seed IDs.
STAT_MODE = "paired"
ALPHA = 0.05
PAIRWISE_CORRECTION = "holm"

# Plot appearance.
FIGSIZE = (10, 6)
BAR_WIDTH = 0.85
BAR_ALPHA = 0.80
ERRORBAR_COLOR = "#202020"
ERRORBAR_LINEWIDTH = 1.0
ERRORBAR_CAPSIZE = 5

POINT_JITTER = 0.165
POINT_SIZE = 31
POINT_ALPHA = 0.88
POINT_EDGE_COLOR = "black"
POINT_EDGE_WIDTH = 0.45
POINT_RANDOM_SEED = 123

DRAW_SEED_LINES = False  # useful for paired seed designs, but can be visually busy.
ROTATE_XTICKS = 0
GRID_ALPHA = 0.28

# Significance annotation style.
SHOW_PAIRWISE_STAR_ANNOTATIONS = True
STAR_ANNOTATE_NS = True
STAR_FONT_SIZE = 10.0
STAR_COLOR = "#222222"
STAR_TEXT_OFFSET_FRACTION = 0.045
STAR_Y_EXTRA_FRACTION = 0.16
SHOW_SIGNIFICANCE_NOTE = True

# Bar value labels.
SHOW_BAR_VALUE_LABELS = True
BAR_VALUE_FORMAT = ".4f"
BAR_VALUE_LABEL_SIZE = 10
BAR_VALUE_LABEL_COLOR = "#222222"
BAR_VALUE_LABEL_OFFSET_FRACTION = 0.025



# ============================================================
# Strategy labels and colors
# ============================================================

STRATEGY_PLOT_LABELS = {
    "S0_naive_mean_control_reference": "Naive\nmean\ncontrol",
    "S1_random_single_control": "Random\nsingle\ncontrol",
    "S2_random_average_controls": "Random\naverage\ncontrol",
    "S4_SEACell_balanced_random_sample": "Metacell\nbalanced\nrandom",
    "S3_SEACell_metacell_average": "Random\nmetacell\naverage",
    "S5_SEACell_OT_sampled_average": "Metacell OT\nsampled\naverage",
}

STRATEGY_BASE_COLORS = {
    "S0_naive_mean_control_reference": "#D0E0EF",
    "S1_random_single_control": "#6E8FB2",
    "S2_random_average_controls": "#7DA494",
    "S3_SEACell_metacell_average": "#E5A79A",
    "S4_SEACell_balanced_random_sample": "#EAB67A",
    "S5_SEACell_OT_sampled_average": "#9F8DB8",
}

# Same variant-aware palette used in the scatter plots.
S5_VARIANT_COLORS = {
    "50&5": "#F5C1D9",
    "200&5": "#D49AB5",
    "350&5": "#9F8DB8",
    "500&5": "#B66699",
}

DEFAULT_STRATEGY_RENAME_MAP = {
    "S0_naive_mean_control_reference": "S0_naive_mean_control_reference",
    "S1_random_single_control": "S1_random_single_control",
    "S2_random_average_controls": "S2_random_average_controls",
    "S3_SEACell_metacell_average": "S3_SEACell_metacell_average",
    "S4_SEACell_balanced_random_sample": "S4_SEACell_balanced_random_sample",
    "S5_SEACell_OT_sampled_average": "S5_SEACell_OT_sampled_average",
    "S0": "S0_naive_mean_control_reference",
    "S1": "S1_random_single_control",
    "S2": "S2_random_average_controls",
    "S3": "S3_SEACell_metacell_average",
    "S4": "S4_SEACell_balanced_random_sample",
    "S5": "S5_SEACell_OT_sampled_average",
    "S4_random_single_control_oracle": "S1_random_single_control",
    "S4_random_single_control": "S1_random_single_control",
    "strategy4_random_single_control": "S1_random_single_control",
    "strategy4_random_single_control_cell": "S1_random_single_control",
    "S3_random_average_controls": "S2_random_average_controls",
    "strategy3_random_average_controls": "S2_random_average_controls",
    "strategy3_random_average_control_cells": "S2_random_average_controls",
    "S5_random_metacell_average": "S3_SEACell_metacell_average",
    "S3_random_metacell_average": "S3_SEACell_metacell_average",
    "strategy5_random_metacell_average": "S3_SEACell_metacell_average",
    "S1_SEACell_balanced_random": "S4_SEACell_balanced_random_sample",
    "S4_SEACell_balanced_random": "S4_SEACell_balanced_random_sample",
    "strategy1_seacell_balanced_random_repeated": "S4_SEACell_balanced_random_sample",
    "S2_SEACell_OT_topk_sampled_average": "S5_SEACell_OT_sampled_average",
    "S2_SEACell_OT_topk_sampled_average_repeated": "S5_SEACell_OT_sampled_average",
    "strategy2_seacells_ot_topk_sampled_average": "S5_SEACell_OT_sampled_average",
    "strategy2_seacell_ot_topk_sampled_average_repeated": "S5_SEACell_OT_sampled_average",
}

STRATEGY_ORDER_MAP = {
    "S0_naive_mean_control_reference": 0,
    "S1_random_single_control": 1,
    "S2_random_average_controls": 2,
    "S3_SEACell_metacell_average": 3,
    "S4_SEACell_balanced_random_sample": 4,
    "S5_SEACell_OT_sampled_average": 5,
}



# ============================================================
# Basic IO helpers
# ============================================================

def read_table(path: str | Path, nrows: int | None = None) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, nrows=nrows)
    if suffix in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t", nrows=nrows)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, nrows=nrows)
    if suffix == ".parquet":
        if nrows == 0:
            # Reading the first row is a practical way to obtain parquet columns.
            return pd.read_parquet(path).head(0)
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported table format: {path}")


def get_table_columns(path: str | Path) -> list[str]:
    try:
        return list(read_table(path, nrows=0).columns)
    except Exception:
        return []


def as_bool_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.lower().isin(["true", "1", "yes", "y"])


def is_missing(x: Any) -> bool:
    if x is None:
        return True
    try:
        return bool(pd.isna(x))
    except Exception:
        return False


def is_valid_color(x: Any) -> bool:
    if is_missing(x):
        return False
    x = str(x).strip()
    return bool(x) and x.lower() not in {"nan", "none", "null"}


def fmt_int_like(x: Any) -> str:
    if is_missing(x):
        return "NA"
    try:
        return str(int(round(float(x))))
    except Exception:
        return str(x)


def clean_label(label: str) -> str:
    return " ".join(str(label).replace("\n", " ").split())


def extract_variant_suffix(display_label: str) -> str:
    display_label = str(display_label)
    if "(" in display_label and ")" in display_label:
        return display_label[display_label.rfind("("):].strip()
    return ""


def extract_number_from_text(x: Any, patterns: Iterable[str]) -> float:
    text = str(x)
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            try:
                return float(m.group(1))
            except Exception:
                pass
    return np.nan


def get_dataset_group_title(path: Path) -> str:
    dataset = path.parent.name.replace("_pseudo_pairing_evaluation", "")
    group = path.name
    return f"{dataset} | {group}"


def get_dataset_group(path: Path) -> tuple[str, str]:
    dataset = path.parent.name.replace("_pseudo_pairing_evaluation", "")
    group = path.name
    return dataset, group



# ============================================================
# Variant harmonization and selected-table matching
# ============================================================

def infer_strategy_column(df: pd.DataFrame) -> str:
    for col in ["strategy", "strategy_id", "pairing_strategy"]:
        if col in df.columns:
            return col
    raise KeyError(f"Cannot infer strategy column from columns: {list(df.columns)}")


def fill_numeric_from_candidates(df: pd.DataFrame, target: str, candidates: list[str]) -> None:
    if target not in df.columns:
        df[target] = np.nan
    df[target] = pd.to_numeric(df[target], errors="coerce")
    for col in candidates:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            df[target] = df[target].where(df[target].notna(), vals)


def fill_from_text_patterns(df: pd.DataFrame, target: str, text_cols: list[str], patterns: list[str]) -> None:
    if target not in df.columns:
        df[target] = np.nan
    df[target] = pd.to_numeric(df[target], errors="coerce")
    for col in text_cols:
        if col not in df.columns:
            continue
        vals = df[col].map(lambda x: extract_number_from_text(x, patterns))
        df[target] = df[target].where(df[target].notna(), vals)


def make_variant_label(row: pd.Series | dict[str, Any]) -> str:
    strategy = str(row.get("strategy", ""))
    nmc = row.get("n_metacells", np.nan)
    topk = row.get("top_k", np.nan)
    sampled = row.get("sampled_metacells_k", np.nan)

    if strategy in {
        "S0_naive_mean_control_reference",
        "S1_random_single_control",
        "S2_random_average_controls",
    }:
        return "default"

    if strategy == "S3_SEACell_metacell_average":
        parts = []
        if not is_missing(nmc):
            parts.append(f"nmc_{fmt_int_like(nmc)}")
        if not is_missing(sampled):
            parts.append(f"sampledMC_{fmt_int_like(sampled)}")
        return "__".join(parts) if parts else "default"

    if strategy == "S4_SEACell_balanced_random_sample":
        return f"nmc_{fmt_int_like(nmc)}" if not is_missing(nmc) else "default"

    if strategy == "S5_SEACell_OT_sampled_average":
        parts = []
        if not is_missing(nmc):
            parts.append(f"nmc_{fmt_int_like(nmc)}")
        if not is_missing(topk):
            parts.append(f"topk_{fmt_int_like(topk)}")
        return "__".join(parts) if parts else "default"

    return "default"


def make_display_variant_label(row: pd.Series | dict[str, Any]) -> str:
    strategy = str(row.get("strategy", ""))
    nmc = row.get("n_metacells", np.nan)
    topk = row.get("top_k", np.nan)
    sampled = row.get("sampled_metacells_k", np.nan)

    if strategy == "S3_SEACell_metacell_average" and not is_missing(nmc) and not is_missing(sampled):
        return f"{strategy} ({fmt_int_like(nmc)}&{fmt_int_like(sampled)})"
    if strategy == "S4_SEACell_balanced_random_sample" and not is_missing(nmc):
        return f"{strategy} ({fmt_int_like(nmc)})"
    if strategy == "S5_SEACell_OT_sampled_average" and not is_missing(nmc) and not is_missing(topk):
        return f"{strategy} ({fmt_int_like(nmc)}&{fmt_int_like(topk)})"
    return strategy


def make_variant_id(row: pd.Series | dict[str, Any]) -> str:
    strategy = str(row.get("strategy", ""))
    label = make_variant_label(row)
    return strategy if label == "default" else f"{strategy}__{label}"


def canonicalize_variants(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    strategy_col = infer_strategy_column(out)
    out["strategy_old"] = out[strategy_col].astype(str)
    out["strategy"] = out["strategy_old"].map(DEFAULT_STRATEGY_RENAME_MAP).fillna(out["strategy_old"])
    out["strategy_order"] = out["strategy"].map(STRATEGY_ORDER_MAP).fillna(99).astype(int)

    fill_numeric_from_candidates(out, "n_metacells", ["n_metacells_requested", "n_metacells_observed"])
    fill_numeric_from_candidates(out, "n_metacells_requested", ["n_metacells"])
    fill_numeric_from_candidates(out, "n_metacells_observed", ["n_metacells"])
    fill_numeric_from_candidates(out, "top_k", ["top_k_metacells"])
    fill_numeric_from_candidates(out, "top_k_metacells", ["top_k"])
    fill_numeric_from_candidates(out, "sampled_metacells_k", ["n_metacells_to_average"])
    fill_numeric_from_candidates(out, "n_metacells_to_average", ["sampled_metacells_k"])
    fill_numeric_from_candidates(out, "n_control_cells_to_average", [])
    fill_numeric_from_candidates(out, "sample_cells_per_metacell", [])

    text_cols = [
        c for c in ["parameter_label", "seacell_setting_id", "run_id", "strategy_old", "display_variant_label", "variant_id"]
        if c in out.columns
    ]
    fill_from_text_patterns(out, "n_metacells", text_cols, [r"nmc[_=-]?(\d+)", r"metacell[s]?[_=-]?(\d+)"])
    fill_from_text_patterns(out, "n_metacells_requested", text_cols, [r"nmc[_=-]?(\d+)"])
    fill_from_text_patterns(out, "top_k", text_cols, [r"topk[_=-]?(\d+)", r"top_k[_=-]?(\d+)"])
    fill_from_text_patterns(out, "top_k_metacells", text_cols, [r"topk[_=-]?(\d+)", r"top_k[_=-]?(\d+)"])
    fill_from_text_patterns(
        out,
        "sampled_metacells_k",
        text_cols,
        [r"sampledMC[_=-]?(\d+)", r"sampled_metacells[_=-]?(\d+)", r"(?:^|__)k[_=-]?(\d+)"],
    )
    fill_from_text_patterns(
        out,
        "n_metacells_to_average",
        text_cols,
        [r"sampledMC[_=-]?(\d+)", r"sampled_metacells[_=-]?(\d+)", r"(?:^|__)k[_=-]?(\d+)"],
    )

    out["variant_label"] = out.apply(make_variant_label, axis=1)
    out["display_variant_label"] = out.apply(make_display_variant_label, axis=1)
    out["variant_id"] = out.apply(make_variant_id, axis=1)
    return out


def get_final_label(row: pd.Series) -> str:
    for col in ["final_strategy_label", "plot_label"]:
        if col in row.index and not is_missing(row[col]) and str(row[col]).strip():
            return clean_label(str(row[col]))
    if "display_variant_label" in row.index and not is_missing(row["display_variant_label"]):
        display = str(row["display_variant_label"])
        suffix = extract_variant_suffix(display)
        base = STRATEGY_PLOT_LABELS.get(str(row.get("strategy", "")), str(row.get("strategy", "")))
        return clean_label(f"{base} {suffix}" if suffix else base)
    return clean_label(str(row.get("variant_id", row.get("strategy", "variant"))))


def choose_color(row: pd.Series) -> str:
    for col in ["manual_color", "color", "plot_color"]:
        if col in row.index and is_valid_color(row[col]):
            return str(row[col]).strip()

    strategy = str(row.get("strategy", ""))
    display = str(row.get("display_variant_label", ""))
    if strategy == "S5_SEACell_OT_sampled_average":
        for key, color in S5_VARIANT_COLORS.items():
            if key in display:
                return color
    return STRATEGY_BASE_COLORS.get(strategy, "#999999")


def load_selected_variants(selection_path: str | Path) -> pd.DataFrame:
    selected = read_table(selection_path)
    selected = canonicalize_variants(selected)
    if "select_for_final" in selected.columns:
        selected = selected[as_bool_series(selected["select_for_final"])].copy()
    if not INCLUDE_S0_AS_BAR:
        selected = selected[selected["strategy"] != "S0_naive_mean_control_reference"].copy()
    if selected.empty:
        raise RuntimeError(f"No selected variants found in {selection_path}")

    selected["final_label"] = selected.apply(get_final_label, axis=1)
    selected["plot_color"] = selected.apply(choose_color, axis=1)

    selected = selected.sort_values(
        ["strategy_order", "n_metacells", "top_k", "sampled_metacells_k", "variant_id"],
        na_position="first",
    ).drop_duplicates("variant_id", keep="first")
    return selected.reset_index(drop=True)



# ============================================================
# Metric-table discovery and loading
# ============================================================

def metric_present_in_columns(columns: Iterable[str]) -> bool:
    cols = set(map(str, columns))
    for canonical, fallbacks in METRIC_SOURCE_FALLBACKS.items():
        if any(col in cols for col in fallbacks):
            return True
    return False


def seed_column_candidates() -> list[str]:
    return [
        "sampling_seed",
        "sampling_seed_for_plot",
        "seed",
        "pair_selection_seed",
        "random_seed",
        "run_seed",
        "mlp_seed",
        "model_seed",
        "split_seed",
        "repeat",
        "run_id",
    ]


def find_existing_seed_column(df: pd.DataFrame) -> str | None:
    for col in seed_column_candidates():
        if col in df.columns:
            return col
    return None


def candidate_metric_paths(sub_path: Path) -> list[Path]:
    """
    Return likely inverse-MLP metric tables.

    Run-level files are listed before result-analysis summary files. The previous
    notebook only searched result_analysis and accidentally chose
    mlp_inverse_seed_averaged_by_strategy_variant_wide.csv. Despite containing
    the word "seed", that file has already averaged the five seeds and has only
    one row per variant.
    """
    result_analysis = sub_path / "result_analysis"
    downstream_mlp = sub_path / "downstream_mlp"
    downstream_task_models = sub_path / "downstream_mlp_task_models"

    common = [
        # Original expression-MLP run-level outputs.
        downstream_mlp / "inverse_mlp_run_summary.csv",
        downstream_mlp / "inverse_mlp_run_summary_PARTIAL.csv",
        downstream_mlp / "inverse_mlp_repeated_run_summary.csv",
        downstream_mlp / "inverse_mlp_repeated_strategy_delta_classification_summary.csv",

        # Representation/task-model run-level output, when applicable.
        downstream_task_models / "combined_inverse_mlp_run_summary.csv",

        # Canonical analysis inputs. These can be run-level or summary-level,
        # so their actual replication counts are inspected below.
        result_analysis / "aggregated_by_task" / "mlp_inverse" / "mlp_inverse_canonical_input_with_required_metrics.csv",
        result_analysis / "aggregated_by_task" / "mlp_inverse" / "inverse_mlp_canonical_input_with_required_metrics.csv",
        result_analysis / "aggregated_by_task" / "inverse_mlp" / "inverse_mlp_canonical_input_with_required_metrics.csv",
        result_analysis / "aggregated_by_task" / "mlp" / "mlp_inverse_canonical_input_with_required_metrics.csv",
        result_analysis / "aggregated_by_task" / "mlp_downstream" / "mlp_inverse_canonical_input_with_required_metrics.csv",
        result_analysis / "mlp_inverse_seed_metrics.csv",
        result_analysis / "inverse_mlp_seed_metrics.csv",
        result_analysis / "mlp_inverse_metrics_by_seed.csv",

        # Summary-only fallbacks. These are retained for diagnostics/error bars,
        # but are strongly deprioritized because they cannot provide five dots.
        result_analysis / "aggregated_by_task" / "mlp_inverse" / "mlp_inverse_seed_averaged_by_strategy_variant_wide.csv",
        result_analysis / "mlp_downstream_seed_averaged_by_strategy_variant_wide.csv",
        result_analysis / "selected_variants_TEMPLATE_EDIT_ME.csv",
    ]

    out: list[Path] = []
    seen: set[Path] = set()

    for p in common:
        if p.exists() and p not in seen:
            out.append(p)
            seen.add(p)

    # Search the relevant result folders, including downstream_mlp, rather than
    # restricting the search to result_analysis.
    search_roots = [result_analysis, downstream_mlp, downstream_task_models]
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for ext in ("*.csv", "*.tsv", "*.txt", "*.xlsx", "*.xls", "*.parquet"):
            for p in search_root.rglob(ext):
                if p in seen:
                    continue
                name = p.name.lower()
                if not any(k in name for k in ["mlp", "inverse", "selected_variants"]):
                    continue
                out.append(p)
                seen.add(p)

    return out


def add_standard_inverse_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for metric, sources in METRIC_SOURCE_FALLBACKS.items():
        if metric not in out.columns:
            out[metric] = np.nan
        out[metric] = pd.to_numeric(out[metric], errors="coerce")

        for src in sources:
            if src in out.columns:
                vals = pd.to_numeric(out[src], errors="coerce")
                out[metric] = out[metric].where(out[metric].notna(), vals)

        std_col = f"{metric}__std_for_errorbar"
        out[std_col] = np.nan
        for src in METRIC_STD_FALLBACKS.get(metric, []):
            if src in out.columns:
                vals = pd.to_numeric(out[src], errors="coerce")
                out[std_col] = out[std_col].where(out[std_col].notna(), vals)

    return out


def inspect_metric_candidate(
    path: Path,
    selected_ids: list[str],
) -> dict[str, Any] | None:
    """
    Inspect a candidate using its actual rows, not just its filename.

    The decisive quantity is the number of distinct run/seed IDs available for
    each selected variant. Summary-only tables typically have exactly one.
    """
    try:
        raw = read_table(path)
    except Exception:
        return None

    if raw.empty or not metric_present_in_columns(raw.columns):
        return None

    try:
        raw = canonicalize_variants(raw)
    except Exception:
        return None

    raw = add_standard_inverse_metrics(raw)

    wanted = set(selected_ids) | {NAIVE_BASELINE_ID, REFERENCE_STRATEGY_FOR_TEST}
    raw = raw[raw["variant_id"].astype(str).isin(wanted)].copy()
    if raw.empty:
        return None

    seed_col = find_existing_seed_column(raw)

    if seed_col is not None:
        # Treat seed/run identifiers as strings so values such as seed_000 and 0
        # remain valid identifiers.
        seed_values = raw[seed_col].astype(str)
        valid_seed = raw[seed_col].notna() & ~seed_values.str.lower().isin(["nan", "none", ""])
        raw = raw.loc[valid_seed].copy()
        rep_counts = raw.groupby("variant_id")[seed_col].nunique(dropna=True)
    else:
        rep_counts = raw.groupby("variant_id").size()

    # Evaluate replication among selected bar variants only. S0 commonly has one
    # deterministic row and should not force the candidate score down.
    selected_rep_counts = rep_counts[rep_counts.index.astype(str).isin(selected_ids)]

    if selected_rep_counts.empty:
        min_reps = median_reps = max_reps = 0.0
    else:
        min_reps = float(selected_rep_counts.min())
        median_reps = float(selected_rep_counts.median())
        max_reps = float(selected_rep_counts.max())

    metric_non_null = 0
    for metric in INVERSE_MLP_METRICS:
        if metric in raw.columns and raw[metric].notna().any():
            metric_non_null += 1

    lower_name = str(path).lower()
    summary_only_name = any(
        token in lower_name
        for token in [
            "seed_averaged",
            "seed-averaged",
            "averaged_by_strategy",
            "variant_wide",
            "selected_variants",
        ]
    )

    # Larger tuples are better.
    score = (
        int(median_reps >= 2),
        median_reps,
        min_reps,
        max_reps,
        int(seed_col is not None),
        metric_non_null,
        -int(summary_only_name),
        -len(str(path)),
    )

    return {
        "path": path,
        "seed_col": seed_col,
        "min_reps": min_reps,
        "median_reps": median_reps,
        "max_reps": max_reps,
        "metric_non_null": metric_non_null,
        "summary_only_name": summary_only_name,
        "score": score,
        "rep_counts": rep_counts.to_dict(),
    }


def find_inverse_mlp_metric_table(
    sub_path: Path,
    selected: pd.DataFrame,
) -> tuple[Path, pd.DataFrame]:
    candidates = candidate_metric_paths(sub_path)
    selected_ids = selected["variant_id"].astype(str).tolist()

    inspected: list[dict[str, Any]] = []
    for p in candidates:
        result = inspect_metric_candidate(p, selected_ids)
        if result is not None:
            inspected.append(result)

    if not inspected:
        tried = "\n".join(str(p) for p in candidates[:80])
        raise FileNotFoundError(
            "Cannot find a table containing usable inverse MLP metrics for the "
            "selected variants. Tried candidates such as:\n" + tried
        )

    inspected = sorted(inspected, key=lambda x: x["score"], reverse=True)

    diagnostics = pd.DataFrame(
        [
            {
                "path": str(x["path"]),
                "seed_column": x["seed_col"],
                "minimum_replicates": x["min_reps"],
                "median_replicates": x["median_reps"],
                "maximum_replicates": x["max_reps"],
                "n_metrics_found": x["metric_non_null"],
                "summary_only_filename": x["summary_only_name"],
                "replicate_counts": str(x["rep_counts"]),
            }
            for x in inspected
        ]
    )

    print("\n[Metric-table candidates, best first]")
    print(
        diagnostics[
            [
                "path",
                "seed_column",
                "minimum_replicates",
                "median_replicates",
                "maximum_replicates",
                "summary_only_filename",
            ]
        ].head(15).to_string(index=False)
    )

    chosen = inspected[0]
    chosen_path = Path(chosen["path"])

    if REQUIRE_REPLICATE_LEVEL_TABLE and chosen["median_reps"] < 2:
        diagnostic_path = (
            sub_path
            / "result_analysis"
            / OUTPUT_FOLDER_NAME
            / "inverse_mlp_metric_table_candidate_diagnostics.csv"
        )
        diagnostic_path.parent.mkdir(parents=True, exist_ok=True)
        diagnostics.to_csv(diagnostic_path, index=False)

        raise RuntimeError(
            "Only summary-level inverse MLP tables were found. The best table "
            f"has median_replicates={chosen['median_reps']:.0f}: {chosen_path}\n"
            "A mean/std/n table cannot reconstruct the five individual run values. "
            "Point the notebook to downstream_mlp/inverse_mlp_run_summary.csv or "
            "another run-level table containing one row per variant and sampling seed.\n"
            f"Candidate diagnostics were saved to: {diagnostic_path}"
        )

    print(
        f"[Metric table] {chosen_path}\n"
        f"[Replicates] seed column={chosen['seed_col']!r}; "
        f"min/median/max={chosen['min_reps']:.0f}/"
        f"{chosen['median_reps']:.0f}/{chosen['max_reps']:.0f}"
    )
    return chosen_path, diagnostics


def infer_seed_column(df: pd.DataFrame) -> str:
    col = find_existing_seed_column(df)
    if col is not None:
        return col

    if REQUIRE_REPLICATE_LEVEL_TABLE:
        raise KeyError(
            "The selected inverse MLP table has no seed/run identifier. Expected "
            f"one of: {seed_column_candidates()}"
        )

    df["_pseudo_seed_index"] = df.groupby("variant_id").cumcount()
    return "_pseudo_seed_index"


def load_inverse_mlp_seed_metrics(
    sub_path: Path,
    selected: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, Path]:
    metric_path, candidate_diagnostics = find_inverse_mlp_metric_table(sub_path, selected)

    outdir = sub_path / "result_analysis" / OUTPUT_FOLDER_NAME
    outdir.mkdir(parents=True, exist_ok=True)
    candidate_diagnostics.to_csv(
        outdir / "inverse_mlp_metric_table_candidate_diagnostics.csv",
        index=False,
    )

    raw = read_table(metric_path)
    raw = canonicalize_variants(raw)
    raw = add_standard_inverse_metrics(raw)
    seed_col = infer_seed_column(raw)

    # If a combined representation table is supplied, avoid silently mixing
    # different representation models within the same variant/seed point.
    if "mlp_representation" in raw.columns:
        representations = raw["mlp_representation"].dropna().astype(str).unique().tolist()
        if len(representations) > 1:
            raise RuntimeError(
                "The selected table contains multiple mlp_representation values: "
                f"{representations}. Use a single-model run-level table or filter "
                "mlp_representation explicitly before plotting."
            )

    metric_cols = list(INVERSE_MLP_METRICS.keys())
    std_cols = [f"{m}__std_for_errorbar" for m in metric_cols]

    keep_cols = [
        "variant_id",
        "strategy",
        "strategy_order",
        "display_variant_label",
        "n_metacells",
        "top_k",
        "sampled_metacells_k",
        seed_col,
    ] + [m for m in metric_cols if m in raw.columns] + [
        c for c in std_cols if c in raw.columns
    ]
    keep_cols = [c for c in keep_cols if c in raw.columns]

    raw = raw[keep_cols].copy()
    raw = raw.rename(columns={seed_col: "sampling_seed_for_plot"})

    # Keep selected variants for bars, S0 for the dashed baseline, and S1 for
    # statistical comparisons.
    selected_ids = selected["variant_id"].astype(str).tolist()
    wanted_ids = selected_ids.copy()
    for extra_id in [NAIVE_BASELINE_ID, REFERENCE_STRATEGY_FOR_TEST]:
        if extra_id not in wanted_ids:
            wanted_ids.append(extra_id)

    seed_df = raw[raw["variant_id"].astype(str).isin(wanted_ids)].copy()

    missing_ids = sorted(set(selected_ids) - set(seed_df["variant_id"].astype(str)))
    missing = selected[selected["variant_id"].isin(missing_ids)].copy()

    seed_df = seed_df.merge(
        selected[["variant_id", "final_label", "plot_color"]],
        on="variant_id",
        how="left",
    )

    s0_mask = seed_df["variant_id"].astype(str) == NAIVE_BASELINE_ID
    if s0_mask.any():
        seed_df.loc[s0_mask, "final_label"] = STRATEGY_PLOT_LABELS.get(
            NAIVE_BASELINE_ID, "Naive average control"
        )
        seed_df.loc[s0_mask, "plot_color"] = STRATEGY_BASE_COLORS.get(
            NAIVE_BASELINE_ID, "#BDBDBD"
        )

    # One plotting point per variant and seed/run. This aggregation only removes
    # accidental duplicate rows within the same seed; it does not average across
    # different seeds.
    group_cols = ["variant_id", "sampling_seed_for_plot"]
    meta_cols = [
        "strategy",
        "strategy_order",
        "display_variant_label",
        "final_label",
        "plot_color",
    ]
    agg = {m: "mean" for m in metric_cols if m in seed_df.columns}
    for c in std_cols:
        if c in seed_df.columns:
            agg[c] = "mean"
    for c in meta_cols:
        if c in seed_df.columns:
            agg[c] = "first"

    seed_df = seed_df.groupby(group_cols, dropna=False, as_index=False).agg(agg)

    replicate_counts = (
        seed_df[seed_df["variant_id"].astype(str).isin(selected_ids)]
        .groupby("variant_id")["sampling_seed_for_plot"]
        .nunique(dropna=True)
        .reindex(selected_ids)
    )

    replicate_report = (
        replicate_counts.rename("n_replications_found")
        .reset_index()
        .merge(
            selected[["variant_id", "final_label"]],
            on="variant_id",
            how="left",
        )
    )
    replicate_report["expected_replications"] = EXPECTED_REPLICATIONS_PER_VARIANT
    replicate_report.to_csv(
        outdir / "inverse_mlp_replicate_count_report.csv",
        index=False,
    )

    print("\n[Replicate counts used for plotting]")
    print(replicate_report.to_string(index=False))

    low_rep = replicate_report[
        replicate_report["n_replications_found"].fillna(0)
        < EXPECTED_REPLICATIONS_PER_VARIANT
    ]
    if not low_rep.empty:
        labels = ", ".join(
            f"{r.final_label}: {int(r.n_replications_found) if pd.notna(r.n_replications_found) else 0}"
            for r in low_rep.itertuples()
        )
        print(
            f"[Warning] Expected {EXPECTED_REPLICATIONS_PER_VARIANT} replications, "
            f"but found fewer for: {labels}"
        )

    if REQUIRE_REPLICATE_LEVEL_TABLE and replicate_counts.fillna(0).max() <= 1:
        raise RuntimeError(
            "The loaded metric table still provides only one value per selected "
            "variant. Individual replicate dots cannot be recovered from a "
            "seed-averaged mean/std table."
        )

    return seed_df, missing, metric_path



# ============================================================
# Statistical testing
# ============================================================

def infer_paired_design(seed_df: pd.DataFrame, metric: str, variant_order: list[str]) -> bool:
    sets = []
    for vid in variant_order:
        vals = seed_df[(seed_df["variant_id"] == vid) & seed_df[metric].notna()]
        sets.append(set(vals["sampling_seed_for_plot"].tolist()))
    if not sets:
        return False
    first = sets[0]
    return len(first) >= 2 and all(s == first for s in sets)


def holm_adjust(p_values: list[float]) -> list[float]:
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    if m == 0:
        return []
    order = np.argsort(p)
    adjusted = np.empty(m, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        adj = (m - rank) * p[idx]
        adj = max(adj, running_max)
        adjusted[idx] = min(adj, 1.0)
        running_max = adjusted[idx]
    return adjusted.tolist()


def p_to_stars(p: float) -> str:
    if pd.isna(p):
        return "ns"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def pairwise_vs_reference_tests(
    seed_df: pd.DataFrame,
    metric: str,
    variant_order: list[str],
    reference_variant: str = REFERENCE_STRATEGY_FOR_TEST,
) -> tuple[pd.DataFrame, str]:
    data = seed_df[seed_df["variant_id"].astype(str).isin(variant_order + [reference_variant])].copy()
    data[metric] = pd.to_numeric(data[metric], errors="coerce")

    if reference_variant not in set(data["variant_id"].astype(str)):
        return pd.DataFrame(), f"No {REFERENCE_STRATEGY_FOR_TEST_LABEL} reference found"

    stat_mode = STAT_MODE.lower()
    if stat_mode not in {"auto", "paired", "unpaired"}:
        raise ValueError("STAT_MODE must be 'auto', 'paired', or 'unpaired'.")

    records: list[dict[str, Any]] = []
    reference_values = data.loc[data["variant_id"] == reference_variant].copy()
    ref_by_seed = reference_values.pivot_table(index="sampling_seed_for_plot", values=metric, aggfunc="mean")

    for i, vid in enumerate(variant_order):
        vid = str(vid)

        if vid == reference_variant:
            records.append({
                "metric": metric,
                "reference_group": reference_variant,
                "group2": vid,
                "group2_index": i,
                "test": "reference",
                "paired": np.nan,
                "n_used": int(reference_values[metric].notna().sum()),
                "mean_reference": float(reference_values[metric].mean()),
                "mean_group2": float(reference_values[metric].mean()),
                "mean_difference_group2_minus_reference": 0.0,
                "statistic": np.nan,
                "p_value": np.nan,
                "p_adj_holm": np.nan,
                "significance": REFERENCE_BAR_TEXT,
                "significant": False,
            })
            continue

        candidate = data.loc[data["variant_id"] == vid].copy()
        if candidate.empty:
            records.append({
                "metric": metric,
                "reference_group": reference_variant,
                "group2": vid,
                "group2_index": i,
                "test": "missing candidate data",
                "paired": False,
                "n_used": 0,
                "mean_reference": float(reference_values[metric].mean()),
                "mean_group2": np.nan,
                "mean_difference_group2_minus_reference": np.nan,
                "statistic": np.nan,
                "p_value": np.nan,
            })
            continue

        cand_by_seed = candidate.pivot_table(index="sampling_seed_for_plot", values=metric, aggfunc="mean")
        paired_frame = ref_by_seed.join(cand_by_seed, how="inner", lsuffix="_ref", rsuffix="_cand").dropna()

        if stat_mode == "paired":
            paired_used = True
        elif stat_mode == "unpaired":
            paired_used = False
        else:
            paired_used = paired_frame.shape[0] >= 2

        if paired_used:
            n = int(paired_frame.shape[0])
            test = "paired t-test vs random single control"
            if n < 2:
                stat = p = np.nan
                mean_ref = float(paired_frame[f"{metric}_ref"].mean()) if n > 0 else np.nan
                mean_cand = float(paired_frame[f"{metric}_cand"].mean()) if n > 0 else np.nan
                mean_diff = mean_cand - mean_ref if n > 0 else np.nan
            else:
                a = paired_frame[f"{metric}_ref"].to_numpy(dtype=float)
                b = paired_frame[f"{metric}_cand"].to_numpy(dtype=float)
                diff = b - a
                mean_ref = float(np.mean(a))
                mean_cand = float(np.mean(b))
                mean_diff = float(np.mean(diff))
                if np.allclose(diff, 0):
                    stat, p = 0.0, 1.0
                else:
                    diff_sd = float(np.std(diff, ddof=1)) if n > 1 else np.nan
                    if np.isfinite(diff_sd) and np.isclose(diff_sd, 0.0) and not np.isclose(mean_diff, 0.0):
                        stat = np.inf if mean_diff > 0 else -np.inf
                        p = 0.0
                    else:
                        stat, p = stats.ttest_rel(b, a, nan_policy="omit", alternative="two-sided")
        else:
            a = reference_values[metric].dropna().to_numpy(dtype=float)
            b = candidate[metric].dropna().to_numpy(dtype=float)
            n = int(min(len(a), len(b)))
            test = "Welch t-test vs random single control"
            mean_ref = float(np.mean(a)) if len(a) > 0 else np.nan
            mean_cand = float(np.mean(b)) if len(b) > 0 else np.nan
            mean_diff = mean_cand - mean_ref if np.isfinite(mean_ref) and np.isfinite(mean_cand) else np.nan
            if len(a) < 2 or len(b) < 2:
                stat = p = np.nan
            elif np.allclose(a.mean(), b.mean()) and np.allclose(a.std(ddof=1), 0) and np.allclose(b.std(ddof=1), 0):
                stat, p = 0.0, 1.0
            else:
                stat, p = stats.ttest_ind(b, a, equal_var=False, nan_policy="omit", alternative="two-sided")

        records.append({
            "metric": metric,
            "reference_group": reference_variant,
            "group2": vid,
            "group2_index": i,
            "test": test,
            "paired": paired_used,
            "n_used": n,
            "mean_reference": mean_ref,
            "mean_group2": mean_cand,
            "mean_difference_group2_minus_reference": mean_diff,
            "statistic": float(stat) if not pd.isna(stat) else np.nan,
            "p_value": float(p) if not pd.isna(p) else np.nan,
        })

    pairwise = pd.DataFrame(records)
    if not pairwise.empty:
        valid_mask = pairwise["p_value"].notna()
        adjusted = [np.nan] * len(pairwise)
        adj_valid = holm_adjust(pairwise.loc[valid_mask, "p_value"].tolist())
        for idx, adj in zip(pairwise.index[valid_mask], adj_valid):
            adjusted[idx] = adj
        pairwise["p_adj_holm"] = adjusted
        pairwise["significance"] = pairwise["p_adj_holm"].map(p_to_stars)
        pairwise.loc[pairwise["group2"].astype(str) == reference_variant, "significance"] = REFERENCE_BAR_TEXT
        pairwise["significant"] = pairwise["p_adj_holm"] < ALPHA

    mode_used = (
        f"each selected strategy vs {REFERENCE_STRATEGY_FOR_TEST_LABEL}; "
        f"paired t-test for matched seeds, Welch t-test otherwise; Holm correction"
    )
    return pairwise, mode_used



# ============================================================
# Plotting
# ============================================================

def metric_summary_for_order(
    metric_df: pd.DataFrame,
    metric: str,
    variant_order: list[str],
) -> tuple[list[np.ndarray], np.ndarray, np.ndarray, np.ndarray]:
    values = []
    means = []
    stds = []
    ns = []
    std_col = f"{metric}__std_for_errorbar"

    for vid in variant_order:
        rows = metric_df.loc[metric_df["variant_id"] == vid].copy()
        vals = pd.to_numeric(rows[metric], errors="coerce").dropna().to_numpy(dtype=float)
        values.append(vals)
        means.append(float(np.nanmean(vals)) if len(vals) else np.nan)

        if len(vals) > 1:
            stds.append(float(np.nanstd(vals, ddof=1)))
        elif len(vals) == 1 and std_col in rows.columns:
            precomputed = pd.to_numeric(rows[std_col], errors="coerce").dropna()
            stds.append(float(precomputed.iloc[0]) if len(precomputed) else 0.0)
        else:
            stds.append(0.0)

        ns.append(int(len(vals)))

    return values, np.asarray(means), np.asarray(stds), np.asarray(ns)


def make_bar_xtick_label(row: pd.Series, n: int) -> str:
    strategy = str(row.get("strategy", ""))

    if USE_STRATEGY_PLOT_LABELS_FOR_XTICKS:
        base = STRATEGY_PLOT_LABELS.get(strategy, clean_label(str(row.get("final_label", strategy))))
    else:
        base = clean_label(str(row.get("final_label", strategy)))

    suffix = ""
    if ADD_VARIANT_SUFFIX_TO_XTICKS:
        suffix = extract_variant_suffix(str(row.get("display_variant_label", "")))

    label = base if not suffix else f"{base}\n{suffix}"
    return f"{label}\nn={int(n)}"


def resolve_y_lim(sub_path: Path, metric: str, data_min: float, data_max: float) -> tuple[float, float]:
    dataset, group = get_dataset_group(sub_path)
    override = Y_LIM_BY_DATASET_GROUP_METRIC.get((dataset, group, metric), None)
    metric_default = Y_LIM_BY_METRIC.get(metric, None)
    lim = override if override is not None else metric_default

    if not np.isfinite(data_min):
        data_min = 0.0
    if not np.isfinite(data_max):
        data_max = 1.0
    span = max(float(data_max - data_min), abs(data_max) * 0.08, 1e-8)
    auto_bottom = max(0.0, data_min - Y_MARGIN_FRACTION * span)
    auto_top = data_max + Y_MARGIN_FRACTION * span

    if lim is None:
        return auto_bottom, auto_top

    bottom, top = lim
    if bottom is None:
        bottom = auto_bottom
    if top is None:
        top = auto_top
    return float(bottom), float(top)


def plot_metric_barplot(
    seed_df: pd.DataFrame,
    selected: pd.DataFrame,
    metric: str,
    outdir: str | Path,
    dataset_group_title: str,
    sub_path: Path,
) -> dict[str, Any]:
    info = INVERSE_MLP_METRICS[metric]
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    available_ids = set(seed_df["variant_id"].astype(str))
    variant_order = [v for v in selected["variant_id"].astype(str).tolist() if v in available_ids]
    if not variant_order:
        print(f"[Skip] No selected variants with values for {metric}.")
        return {"metric": metric, "skipped": True}

    selected_lookup = selected.set_index("variant_id")
    selected_plot_rows = selected_lookup.loc[variant_order].copy()

    y_values, means, stds, ns = metric_summary_for_order(seed_df, metric, variant_order)
    colors = [selected_lookup.loc[v, "plot_color"] for v in variant_order]
    positions = np.arange(len(variant_order))

    # S0 naive reference line.
    baseline_values = pd.to_numeric(
        seed_df.loc[seed_df["variant_id"] == NAIVE_BASELINE_ID, metric],
        errors="coerce",
    ).dropna().to_numpy(dtype=float)
    baseline_mean = float(np.nanmean(baseline_values)) if len(baseline_values) else np.nan
    baseline_std = float(np.nanstd(baseline_values, ddof=1)) if len(baseline_values) > 1 else np.nan

    pairwise, mode_used = pairwise_vs_reference_tests(seed_df, metric, variant_order)

    fig, ax = plt.subplots(figsize=FIGSIZE)

    ax.bar(
        positions,
        means,
        yerr=stds,
        width=BAR_WIDTH,
        color=colors,
        alpha=BAR_ALPHA,
        edgecolor="black",
        linewidth=0.75,
        error_kw={
            "ecolor": ERRORBAR_COLOR,
            "elinewidth": ERRORBAR_LINEWIDTH,
            "capsize": ERRORBAR_CAPSIZE,
            "capthick": ERRORBAR_LINEWIDTH,
        },
        zorder=2,
    )

    # Overlay individual seed/run dots.
    rng = np.random.default_rng(POINT_RANDOM_SEED)
    for i, (vals, color) in enumerate(zip(y_values, colors)):
        if len(vals) == 0:
            continue
        jitter = rng.uniform(-POINT_JITTER, POINT_JITTER, size=len(vals))
        ax.scatter(
            np.full(len(vals), positions[i]) + jitter,
            vals,
            s=POINT_SIZE,
            color=color,
            alpha=POINT_ALPHA,
            edgecolor=POINT_EDGE_COLOR,
            linewidth=POINT_EDGE_WIDTH,
            zorder=5,
        )

    # Optionally draw paired seed lines.
    if DRAW_SEED_LINES:
        wide = seed_df[seed_df["variant_id"].isin(variant_order)].pivot_table(
            index="sampling_seed_for_plot",
            columns="variant_id",
            values=metric,
            aggfunc="mean",
        )
        for _, row in wide.iterrows():
            if row.notna().sum() < 2:
                continue
            xs, ys = [], []
            for i, vid in enumerate(variant_order):
                if pd.notna(row.get(vid, np.nan)):
                    xs.append(positions[i])
                    ys.append(float(row[vid]))
            ax.plot(xs, ys, color="#999999", alpha=0.25, linewidth=0.7, zorder=1)

    finite_parts = [means[np.isfinite(means)], (means + stds)[np.isfinite(means + stds)]]
    for vals in y_values:
        if len(vals):
            finite_parts.append(vals[np.isfinite(vals)])
    if len(baseline_values):
        finite_parts.append(np.asarray([baseline_mean, baseline_mean + (baseline_std if np.isfinite(baseline_std) else 0.0)]))
    finite_all = np.concatenate([x for x in finite_parts if len(x)]) if finite_parts else np.array([0.0, 1.0])
    data_min = float(np.nanmin(finite_all))
    data_max = float(np.nanmax(finite_all))
    span = max(float(data_max - data_min), abs(data_max) * 0.08, 1e-8)

    # Exact bar-value labels.
    bar_value_y_lookup: dict[int, float] = {}
    if SHOW_BAR_VALUE_LABELS:
        value_format = info.get("value_format", BAR_VALUE_FORMAT)
        for i, (pos, mean_value, std_value, vals) in enumerate(zip(positions, means, stds, y_values)):
            if not np.isfinite(mean_value):
                continue
            y_bar_top = mean_value + (std_value if np.isfinite(std_value) else 0.0)
            y_data_top = np.nanmax(vals) if len(vals) else y_bar_top
            y_text = max(y_bar_top, y_data_top) + BAR_VALUE_LABEL_OFFSET_FRACTION * span
            bar_value_y_lookup[i] = y_text
            ax.text(
                pos,
                y_text,
                f"{mean_value:{value_format}}",
                ha="center",
                va="bottom",
                fontsize=BAR_VALUE_LABEL_SIZE,
                color=BAR_VALUE_LABEL_COLOR,
                clip_on=False,
                zorder=8,
            )

    # S0 naive average control baseline as a horizontal dashed line.
    if SHOW_NAIVE_BASELINE_LINE and np.isfinite(baseline_mean):
        ax.axhline(
            baseline_mean,
            color=NAIVE_BASELINE_LINE_COLOR,
            linestyle=NAIVE_BASELINE_LINESTYLE,
            linewidth=NAIVE_BASELINE_LINEWIDTH,
            alpha=NAIVE_BASELINE_LINE_ALPHA,
            zorder=10,
        )
        x_text = positions[-1] + NAIVE_BASELINE_TEXT_X_FRACTION * BAR_WIDTH
        y_text = baseline_mean + NAIVE_BASELINE_TEXT_Y_OFFSET_FRACTION * span
        line_label = f"{NAIVE_BASELINE_TEXT}: {baseline_mean:.4g}"
        if np.isfinite(baseline_std):
            line_label += f" ± {baseline_std:.2g}"
        ax.text(
            x_text,
            y_text,
            line_label,
            ha="right",
            va="bottom",
            fontsize=NAIVE_BASELINE_TEXT_SIZE,
            color=NAIVE_BASELINE_TEXT_COLOR,
            clip_on=False,
            zorder=10,
        )

    # Pairwise star annotations vs S1.
    y_star_values = []
    if SHOW_PAIRWISE_STAR_ANNOTATIONS and not pairwise.empty:
        star_lookup = {str(row["group2"]): str(row["significance"]) for _, row in pairwise.iterrows()}
        text_offset = STAR_TEXT_OFFSET_FRACTION * span

        for i, vid in enumerate(variant_order):
            label = star_lookup.get(vid, "ns")
            if vid == REFERENCE_STRATEGY_FOR_TEST and not SHOW_REFERENCE_BAR_TEXT:
                continue
            if label == "ns" and not STAR_ANNOTATE_NS:
                continue

            y_bar_top = means[i] + (stds[i] if np.isfinite(stds[i]) else 0.0)
            y_data_top = np.nanmax(y_values[i]) if len(y_values[i]) else y_bar_top
            y_text = max(y_bar_top, y_data_top, bar_value_y_lookup.get(i, -np.inf)) + 0.65 * text_offset
            y_star_values.append(y_text)
            ax.text(
                positions[i],
                y_text,
                label,
                ha="center",
                va="bottom",
                fontsize=STAR_FONT_SIZE,
                color=STAR_COLOR,
                clip_on=False,
                zorder=8,
            )

    label_tops = list(bar_value_y_lookup.values()) + y_star_values
    if label_tops:
        data_max = max(data_max, max(label_tops) + STAR_Y_EXTRA_FRACTION * span)

    bottom, top = resolve_y_lim(sub_path, metric, data_min, data_max)
    # Make sure annotations are not clipped if a fixed top was set too low.
    if label_tops and top < max(label_tops) + 0.05 * span:
        print(f"[Warning] Fixed y-limit top for {metric} may clip annotations: top={top}, needed≈{max(label_tops) + 0.05 * span:.4g}")
    ax.set_ylim(bottom=bottom, top=top)

    ax.set_title(f"{info['label']} across selected variants\n{dataset_group_title}", fontsize=18, weight="bold")
    ax.set_ylabel(info["ylabel"], fontsize=15)
    ax.set_xticks(positions)
    xtick_labels = [make_bar_xtick_label(selected_lookup.loc[v], n) for v, n in zip(variant_order, ns)]
    ax.set_xticklabels(xtick_labels, rotation=ROTATE_XTICKS, ha="center", fontsize=13)
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=GRID_ALPHA)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if SHOW_SIGNIFICANCE_NOTE and SHOW_PAIRWISE_STAR_ANNOTATIONS:
        note = f"Stars/ns: Holm-adjusted tests vs {REFERENCE_STRATEGY_FOR_TEST_LABEL}; S0 shown only as baseline."
        ax.text(0.01, 0.98, note, transform=ax.transAxes, ha="left", va="top", fontsize=10, color="#555555")

    fig.tight_layout()

    fig_base = outdir / info["save_name"]
    if SAVE_PNG:
        png_path = fig_base.with_suffix(".png")
        fig.savefig(png_path, dpi=DPI, bbox_inches="tight")
        print(f"[Saved] {png_path}")
    if SAVE_SVG:
        svg_path = fig_base.with_suffix(".svg")
        fig.savefig(svg_path, bbox_inches="tight")
        print(f"[Saved] {svg_path}")

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)

    return {
        "metric": metric,
        "figure_base": fig_base,
        "stat_mode_used": mode_used,
        "pairwise": pairwise,
        "means": dict(zip(variant_order, means)),
        "stds": dict(zip(variant_order, stds)),
        "ns": dict(zip(variant_order, ns)),
        "naive_baseline_mean": baseline_mean,
        "naive_baseline_std": baseline_std,
        "naive_baseline_n": int(len(baseline_values)),
    }



# ============================================================
# Main execution
# ============================================================

def process_one_sub_path(sub_path: Path) -> dict[str, Any]:
    print(f"\n[Processing] {sub_path}")
    selection_path = sub_path / "result_analysis" / "selected_variants_TEMPLATE_EDIT_ME.csv"
    if not selection_path.exists():
        print(f"[Skip] Missing selection table: {selection_path}")
        return {"sub_path": sub_path, "skipped": True, "reason": "missing selection table"}

    selected = load_selected_variants(selection_path)
    seed_df, missing, metric_path = load_inverse_mlp_seed_metrics(sub_path, selected)

    outdir = sub_path / "result_analysis" / OUTPUT_FOLDER_NAME
    outdir.mkdir(parents=True, exist_ok=True)

    if not missing.empty:
        missing_labels = ", ".join(missing["final_label"].astype(str).tolist())
        print(f"[Warning] Selected variants missing in inverse MLP metric table: {missing_labels}")

    # Warn when the table appears summary-only. Dots will still be shown, but they may represent one summary value per variant.
    count_by_variant = seed_df.groupby("variant_id")["sampling_seed_for_plot"].nunique()
    if len(count_by_variant) and count_by_variant.max() <= 1:
        print("[Warning] Metric table appears summary-only; dots represent available summary rows rather than multiple seed-level runs.")

    outputs = []
    title = get_dataset_group_title(sub_path)
    for metric in INVERSE_MLP_METRICS:
        if metric not in seed_df.columns or seed_df[metric].notna().sum() == 0:
            print(f"[Skip] No values for metric: {metric}")
            continue
        outputs.append(plot_metric_barplot(seed_df, selected, metric, outdir, title, sub_path))

    return {"sub_path": sub_path, "outdir": outdir, "metric_table": metric_path, "outputs": outputs}


def main() -> list[dict[str, Any]]:
    candidate_paths = [root_dir / f"{name}_pseudo_pairing_evaluation" for name in dataset_names]
    detailed_sub_paths = [sub_path / group for sub_path in candidate_paths for group in groups]
    valid_paths = [sub_path for sub_path in detailed_sub_paths if sub_path.exists()]

    if RUN_ALL_VALID_PATHS:
        targets = valid_paths
    else:
        if SELECTED_SUB_PATH is None:
            if not valid_paths:
                raise RuntimeError("No valid dataset/group paths found.")
            targets = [valid_paths[-1]]
        else:
            targets = [Path(SELECTED_SUB_PATH)]

    if not targets:
        raise RuntimeError("No dataset/group paths selected for plotting.")

    all_outputs = []
    for target in targets:
        try:
            all_outputs.append(process_one_sub_path(Path(target)))
        except Exception as e:
            print(f"[Error] Failed on {target}: {e}")
            raise
    return all_outputs



# Run all selected dataset/group paths.
# If you only want one folder, set RUN_ALL_VALID_PATHS = False and SELECTED_SUB_PATH above.
outputs = main()
